# Chat

Mirrors the structure of the LangChain "Chat with Your Data" chat lecture (`06_chat.ipynb`), applied to our real Bahraini legal corpus.

## Overview recap

We've covered document splitting and embeddings (before this notebook), retrieval (`05_retrieval_langchain.ipynb`), and question-answering with `RetrievalQA` (`06_question_answering_langchain.ipynb`) — including that chain's key limitation: it has no memory of earlier turns. This notebook adds memory and wraps everything in an interactive chat interface.

In [ ]:
!pip install -q langchain-huggingface langchain-chroma langchain-core langchain-text-splitters langchain-community langchain-classic langchain-openai sentence-transformers transformers chromadb panel param


In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")


If you want to experiment with the **LangSmith** platform to trace and inspect these chains (this mirrors the lecture's optional LangSmith step — it is off by default and nothing here requires it):

* Go to [LangSmith](https://www.langchain.com/langsmith) and sign up
* Create an API key from your account's settings
* Paste it below and uncomment the cell


In [ ]:
# import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
# os.environ["LANGCHAIN_API_KEY"] = "..."  # replace with your own LangSmith key
# os.environ["LANGCHAIN_PROJECT"] = "capital-legal-base"


In [ ]:
import panel as pn
pn.extension()

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from google.colab import drive
import torch

drive.mount("/content/drive")

persist_directory = "/content/drive/MyDrive/law_chatbot_chroma_v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)
llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

In [ ]:
question = "ما هي شروط استحقاق النفقة الزوجية؟"
docs = vectordb.similarity_search(question, k=3)
print(len(docs))

llm.invoke("مرحبا").content

### Recap: prompt + RetrievalQA chain

In [ ]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

SYSTEM_TEMPLATE = """انت مساعد قانوني متخصص في القانون البحريني. استخدم المقاطع القانونية التالية فقط للاجابة على السؤال في نهاية النص.

قواعد صارمة يجب اتباعها:
- استند فقط الى النصوص المرفقة، ولا تخترع اي معلومة غير موجودة فيها.
- اذا لم تكن الاجابة موجودة في النصوص المرفقة، صرح بذلك بوضوح ولا تخمن.
- اذكر المصدر الدقيق لكل معلومة (رقم المادة او رقم القضية).
- اذا استندت الاجابة الى اكثر من قانون او حكم، اذكرهم جميعا.

النصوص القانونية:
{context}

السؤال: {question}

الاجابة القانونية المدعومة بالمصادر:"""
QA_CHAIN_PROMPT = PromptTemplate.from_template(SYSTEM_TEMPLATE)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
)
result = qa_chain.invoke({"query": question})
result["result"]

### Memory

In [ ]:
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

### ConversationalRetrievalChain

In [ ]:
from langchain_classic.chains import ConversationalRetrievalChain

retriever = vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2})
qa = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
)

In [ ]:
result = qa.invoke({"question": question})
result["answer"]

In [ ]:
follow_up = "ما المدة التي يجب ان يغيبها العامل حتى يجوز فصله؟"
result = qa.invoke({"question": follow_up})
result["answer"]

## Create a chatbot that works on our legal documents

In [ ]:
import param

def load_qa_chain(llm_model="nvidia/nemotron-3-ultra-550b-a55b:free"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device})
    vectordb = Chroma(persist_directory="/content/drive/MyDrive/law_chatbot_chroma_v2", embedding_function=embedding)
    llm = ChatOpenAI(
        model=llm_model,
        temperature=0,
        api_key=os.environ["OPENROUTER_API_KEY"],
        base_url="https://openrouter.ai/api/v1",
    )
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    return ConversationalRetrievalChain.from_llm(
        llm,
        retriever=vectordb.as_retriever(search_type="mmr", search_kwargs={"k": 2}),
        memory=memory,
        return_source_documents=True,
        return_generated_question=True,
        combine_docs_chain_kwargs={"prompt": QA_CHAIN_PROMPT},
    )


class cbfs(param.Parameterized):
    chat_history = param.List([])
    answer = param.String("")
    db_query = param.String("")
    db_response = param.List([])

    def __init__(self, **params):
        super(cbfs, self).__init__(**params)
        self.panels = []
        self.qa = load_qa_chain()

    def convchain(self, query):
        if not query:
            return pn.WidgetBox(pn.Row("المستخدم:", pn.pane.Markdown("", width=600)), scroll=True)
        result = self.qa.invoke({"question": query})
        self.chat_history.extend([(query, result["answer"])])
        self.db_query = result.get("generated_question", "")
        self.db_response = result.get("source_documents", [])
        self.answer = result["answer"]
        self.panels.extend([
            pn.Row("المستخدم:", pn.pane.Markdown(query, width=600)),
            pn.Row("المساعد القانوني:", pn.pane.Markdown(self.answer, width=600, styles={"background-color": "#F6F6F6"})),
        ])
        inp.value = ""
        return pn.WidgetBox(*self.panels, scroll=True)

    @param.depends("db_query")
    def get_lquest(self):
        if not self.db_query:
            return pn.Column(
                pn.Row(pn.pane.Markdown("اخر استعلام لقاعدة البيانات:", styles={"background-color": "#F6F6F6"})),
                pn.Row(pn.pane.Str("لا يوجد استعلام حتى الان"))
            )
        return pn.Column(
            pn.Row(pn.pane.Markdown("استعلام قاعدة البيانات:", styles={"background-color": "#F6F6F6"})),
            pn.pane.Str(self.db_query),
        )

    @param.depends("db_response")
    def get_sources(self):
        if not self.db_response:
            return
        rlist = [pn.Row(pn.pane.Markdown("المصادر المسترجعة:", styles={"background-color": "#F6F6F6"}))]
        for doc in self.db_response:
            rlist.append(pn.Row(pn.pane.Str(f"{doc.metadata} — {doc.page_content[:150]}")))
        return pn.WidgetBox(*rlist, width=600, scroll=True)

    def clr_history(self, count=0):
        self.chat_history = []
        self.panels = []
        return

### Dashboard

In [ ]:
cb = cbfs()

button_clearhistory = pn.widgets.Button(name="مسح المحادثة", button_type="warning")
button_clearhistory.on_click(cb.clr_history)
inp = pn.widgets.TextInput(placeholder="اكتب سؤالك القانوني هنا...")

conversation = pn.bind(cb.convchain, inp)

tab1 = pn.Column(
    pn.Row(inp),
    pn.layout.Divider(),
    pn.panel(conversation, loading_indicator=True, height=400),
)
tab2 = pn.Column(pn.panel(cb.get_lquest))
tab3 = pn.Column(pn.panel(cb.get_sources))
tab4 = pn.Column(
    pn.Row(button_clearhistory, pn.pane.Markdown("يمسح سجل المحادثة لبدء موضوع جديد")),
)

dashboard = pn.Column(
    pn.Row(pn.pane.Markdown("# المساعد القانوني - Capital Legal Base")),
    pn.Tabs(
        ("المحادثة", tab1),
        ("قاعدة البيانات", tab2),
        ("المصادر", tab3),
        ("الاعدادات", tab4),
    ),
)
dashboard

### Note on the demo environment

This Panel dashboard is a lecture-style, notebook-native chat UI — useful for quickly exercising the chain interactively while developing. The actual deployed demo surfaces for this project are the Streamlit app (`08_streamlit_app/app.py`) and the Chainlit app (`08_streamlit_app/appchainlit.py`), which add citation-scope checking, retry-on-overload handling, and a switchable LLM provider on top of the same underlying chain logic shown here.